# Reference copy - needs non-public data

이 노트북은 원본 연구 노트북의 **참고용 사본**입니다. 실행에 필요한 02 노트북의 예측 결과 CSV, 월간 가격 엑셀, 팩터 CSV은(는) 유료/비공개 데이터라 이 저장소에 포함하지 않았으므로 **그대로는 끝까지 실행되지 않습니다.** 출력, 실행 카운트, 로컬 경로는 제거했습니다. 실행 가능한 데모는 `00_public_sample_workflow.ipynb`(합성 데이터)를 사용하세요.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

print("Loading Simulation Results...")

# 1. 데이터 불러오기
# index_col=0: Date를 인덱스로
# parse_dates=True: 날짜 형식 인식
df_ols_wide = pd.read_csv("Result_Pred_OLS.csv", index_col=0, parse_dates=True, encoding='cp949')
df_xgb_wide = pd.read_csv("Result_Pred_XGB.csv", index_col=0, parse_dates=True, encoding='cp949')
df_actual_wide = pd.read_csv("Result_Actual_EP.csv", index_col=0, parse_dates=True, encoding='cp949')

print("Data Loaded.")
print(f"- OLS Shape: {df_ols_wide.shape}")
print(f"- XGB Shape: {df_xgb_wide.shape}")
print(f"- Actual Shape: {df_actual_wide.shape}")

# 2. Mispricing (괴리) 재계산
# 괴리 = 예측 - 실제
mispricing_ols = df_ols_wide - df_actual_wide
mispricing_xgb = df_xgb_wide - df_actual_wide

print("Mispricing Calculated.")

In [ ]:
price = pd.read_excel("price_m.xlsx", index_col=0, parse_dates=True, header=8)

ret = price.iloc[5:].pct_change(fill_method=None)
ret.index = pd.to_datetime(ret.index)
ret = ret.resample("ME").last()
ret = ret.astype(float)

def winsorize_row(row, lower=0.005, upper=0.005):
    
    q_low = row.quantile(lower)
    q_high = row.quantile(1 - upper)
    
    return row.clip(lower=q_low, upper=q_high)

ret = ret.apply(lambda r: winsorize_row(r, 0, 0.005), axis=1)

ret = ret.loc[
    ret.index.intersection(mispricing_ols.index),
    ret.columns.intersection(mispricing_ols.columns)
]

print(f"- Return Shape: {ret.shape}")

In [ ]:
next_ret = ret.copy().shift(-1)

# 모델 예측 결과(mispricing_ols)를 기준으로 형태 강제 일치 (Reindex)
# intersection을 쓰면 미래 예측치(10월)가 날아가지만, 
# reindex를 쓰면 10월 행이 유지되고 값은 NaN이 됨 (이게 더 안전한 방식)
aligned_ret = next_ret.reindex(index=mispricing_ols.index, columns=mispricing_ols.columns)

print("-" * 30)
print(f"Target Shape (Model):  {mispricing_ols.shape}")
print(f"Aligned Return Shape:  {aligned_ret.shape}")
print("-" * 30)

# 데이터 확인 (마지막 부분에 NaN이 있는지 확인)
print("Last 3 rows of Aligned Returns:")
print(aligned_ret.iloc[-3:, :5])

---
## **H1 검증**

In [ ]:
# ---------------------------------------------------------
# H1 검증: 예측 성능 지표 (RMSE, MAE, R2_OOS)
# ---------------------------------------------------------

# 1. Error 계산 (전체 패널을 1줄로 펴서 계산)
# NaN은 제거 (데이터가 없는 구간 제외)
# (df_ols_wide - df_actual_wide)가 곧 mispricing_ols와 같음
err_ols = (mispricing_ols).stack().dropna()
err_xgb = (mispricing_xgb).stack().dropna()

# 2. 성능 지표 계산 함수
def calc_metrics(errors):
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    return mse, rmse, mae

mse_ols, rmse_ols, mae_ols = calc_metrics(err_ols)
mse_xgb, rmse_xgb, mae_xgb = calc_metrics(err_xgb)

# 3. Out-of-Sample R-squared (Campbell & Thompson)
# R2_OOS = 1 - (MSE_Challenger / MSE_Benchmark)
# 양수(+)면 XGB가 더 잘한 것, 음수(-)면 OLS가 더 잘한 것
r2_oos = 1 - (mse_xgb / mse_ols)

# 4. 결과 리포트
h1_report = pd.DataFrame({
    'Metric': ['RMSE', 'MAE'],
    'Linear (OLS)': [rmse_ols, mae_ols],
    'XGBoost (ML)': [rmse_xgb, mae_xgb]
})

print("=== [H1 Result] Valuation Accuracy ===")
print(h1_report)
print("-" * 30)
print(f"Out-of-Sample R2 (ML vs Linear): {r2_oos*100:.4f}%")

if r2_oos > 0:
    print(">> Conclusion: 머신러닝이 선형 모델보다 예측 정확도가 높습니다. (H1 채택)")
else:
    print(">> Conclusion: 머신러닝의 예측력이 선형 모델보다 떨어집니다. (H1 기각 위기)")

In [ ]:
# ---------------------------------------------------------
# H1 결과 출력 (Prettier Output with Highlight)
# ---------------------------------------------------------
print("\n=== [H1 Result] Valuation Accuracy (Error Metrics) ===")

# 1. 스타일링: 소수점 4자리 & 더 작은 에러값(Winner)에 초록색 음영 표시
# axis=1: 행(Row) 단위로 비교 (Linear vs XGBoost)
styled_h1 = h1_report.style.format(precision=4)

# 2. 예쁘게 출력
display(styled_h1)

# 3. 판정 로직
print("-" * 50)
print(f"Out-of-Sample R2 (Campbell-Thompson): {r2_oos*100:.4f}%")

if r2_oos > 0:
    print(f">> [Pass] ML 모델이 선형 모델 대비 MSE를 {r2_oos*100:.2f}% 감소시켰습니다.")
    print(">> 결론: 머신러닝 기반 밸류에이션 모델의 예측 정확도가 통계적으로 우월합니다. (H1 채택)")
else:
    print(f">> [Fail] ML 모델의 예측력이 선형 모델보다 떨어집니다. (R2 < 0)")
    print(">> 결론: H1 가설이 기각될 위기입니다.")
print("-" * 50)

---
## **H3검증 : Fama-MacBeth Regression**
* ML 괴리율과 OLS 괴리율을 경쟁시켜(Horse Race), ML이 추가적인 정보(Incremental Information)를 제공하는지 검증.
* **Model:** $R_{t+1} = \alpha + \beta_1 \text{Mis}_{OLS, t} + \beta_2 \text{Mis}_{XGB, t} + \epsilon$

In [ ]:
import statsmodels.api as sm
from tqdm import tqdm

print("Running H3 Fama-MacBeth Regression...")

# 1. 데이터 병합 (Long Format으로 변환)
# stack()을 하면 (Date, Ticker) 인덱스가 생김
s_ols = mispricing_ols.stack().rename("Mis_OLS")
s_xgb = mispricing_xgb.stack().rename("Mis_XGB")
s_ret = aligned_ret.stack().rename("Return")

# 인덱스 기준으로 합치기 (Inner Join -> 수익률이 없는 미래 데이터는 자동 삭제됨)
reg_df = pd.concat([s_ols, s_xgb, s_ret], axis=1).dropna()

print(f"- Regression Data Points: {len(reg_df):,}")

In [ ]:
reg_df.head()

In [ ]:
# 2. Fama-MacBeth Regression Loop
fmb_coeffs = []

# 매월 횡단면 회귀 수행
for date, group in tqdm(reg_df.groupby('Date')):
    # 최소 관측치 필터링 (종목 수가 너무 적은 달 제외)
    if len(group) < 30: continue
    
    Y = group['Return']
    X = group[['Mis_OLS', 'Mis_XGB']]
    X = sm.add_constant(X)
    
    try:
        model = sm.OLS(Y, X).fit()
        fmb_coeffs.append(model.params)
    except:
        continue

# 3. 결과 집계 (Newey-West t-stat 적용)
fmb_df = pd.DataFrame(fmb_coeffs)

def get_nw_tstat(series, lags=3):
    if len(series) < 10: return np.nan
    # 평균이 0인지 검정 (상수항만 있는 회귀 + HAC Error)
    X = np.ones_like(series)
    res = sm.OLS(series, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return res.tvalues[0]

summary_h3 = pd.DataFrame({
    'Mean Coef': fmb_df.mean(),
    't-stat (NW)': [get_nw_tstat(fmb_df[col]) for col in fmb_df.columns]
})

print("\n=== [H3 Result] Incremental Information Test ===")
print(summary_h3)

if summary_h3.loc['Mis_XGB', 't-stat (NW)'] > 1.96:
    print(">> [Pass] ML 모델이 유의한 추가 정보를 제공합니다. (H3 채택)")
else:
    print(">> [Fail] ML 모델의 설명력이 유의하지 않습니다.")

In [ ]:
# ---------------------------------------------------------
# H3 결과 출력 (Prettier Output)
# ---------------------------------------------------------
print("\n=== [H3 Result] Incremental Information Test (Fama-MacBeth) ===")

# 1. 보기 좋게 스타일링 (소수점 4자리 고정, 중요한 값 강조)
# summary_h3는 DataFrame이어야 함
styled_h3 = summary_h3.style.format("{:.4f}")
# 2. 예쁘게 출력
display(styled_h3)

# 3. 판정 로직
t_stat_xgb = summary_h3.loc['Mis_XGB', 't-stat (NW)']

print("-" * 50)
if t_stat_xgb > 1.96:
    print(f">> [Pass] ML 모델의 t-stat이 {t_stat_xgb:.4f}로 1.96을 상회합니다.")
    print(">> 결론: ML 괴리율은 통계적으로 유의한 추가 정보(Incremental Info)를 제공합니다. (H3 채택)")
elif t_stat_xgb < -1.96:
    print(f">> [Check] ML 모델의 t-stat이 {t_stat_xgb:.4f}로 유의하지만 음수입니다.")
    print(">> 결론: 부호가 반대입니다. 괴리율 정의나 방향성을 다시 확인하세요.")
else:
    print(f">> [Fail] ML 모델의 t-stat이 {t_stat_xgb:.4f}로 유의하지 않습니다.")
    print(">> 결론: ML 모델의 추가 설명력이 부족합니다.")
print("-" * 50)

---
## **H2 검증 : Portfolio Performance**
* ML 괴리율을 기준으로 10분위(Decile) 포트폴리오를 구성.
* **Long-Short Strategy:** Decile 10 (저평가) 매수 - Decile 1 (고평가) 공매도.


In [ ]:
n = 10

long_position = mispricing_xgb.apply(lambda row: row.where(row <= row.nsmallest(n).max()), axis=1)
long_position = (long_position/long_position)

short_position = mispricing_xgb.apply(lambda row: row.where(row >= row.nlargest(n).min()), axis=1)
short_position = -(short_position/short_position)

In [ ]:
port = -(long_position * aligned_ret).dropna(how='all').mean(axis=1)
short_port = (short_position * aligned_ret).dropna(how='all').mean(axis=1)

In [ ]:
long_short_port = (port + short_port)

---

In [ ]:
df_raw = pd.read_csv("portfolio_returns.csv", index_col=0, parse_dates=True)

In [ ]:
np.log1p(df_raw).cumsum().plot()

In [ ]:
np.log1p(df_raw["long_short"]).cumsum().plot()


In [ ]:
np.log1p(df_raw["long"]).cumsum().plot()

In [ ]:
np.log1p(df_raw["short"]).cumsum().plot()

---

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Investment Factor 계산에 필요한 데이터를 input 폴더에서 불러옴
factors   = pd.read_csv("./input/factors_monthly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)

In [ ]:
# 1. 두 데이터 공통 구간 맞추기
df = pd.concat([df_raw, factors], axis=1, join="inner").dropna()

In [ ]:
df.head()

In [ ]:
# 2. 종속변수: 포트폴리오 초과수익률
portfolio = 'long_short'
y = df[portfolio] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})  # Newey-West 표준오차

# 5. 출력
print(model.summary())

In [ ]:
# 2. 종속변수: 포트폴리오 초과수익률
portfolio = 'long'
y = df[portfolio] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})  # Newey-West 표준오차

# 5. 출력
print(model.summary())

In [ ]:
# 2. 종속변수: 포트폴리오 초과수익률
portfolio = 'short'
y = df[portfolio] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})  # Newey-West 표준오차

# 5. 출력
print(model.summary())

In [ ]:
df_raw

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. 성과 지표 계산 함수 (Academic Standard)
# ---------------------------------------------------------
def calculate_performance(series):
    # 연율화 계수 (월간 데이터 기준)
    ann_factor = 12
    sqrt_factor = np.sqrt(12)
    
    # 1. 연환산 수익률 (Annualized Return)
    # 단순 평균 * 12 (Log return이 아닌 Simple return일 경우)
    ann_ret = series.mean() * ann_factor
    
    # 2. 연환산 변동성 (Annualized Volatility)
    ann_vol = series.std() * sqrt_factor
    
    # 3. 샤프 지수 (Sharpe Ratio)
    # Risk-free rate = 0으로 가정
    if ann_vol == 0:
        sharpe = 0
    else:
        sharpe = ann_ret / ann_vol
    
    # 4. MDD (Maximum Drawdown)
    # 누적 수익률(Wealth Index) 계산
    cum_ret = (1 + series).cumprod()
    # 전고점(Running Max) 계산
    running_max = cum_ret.cummax()
    # 낙폭(Drawdown) 계산
    drawdown = (cum_ret - running_max) / running_max
    mdd = drawdown.min()
    
    # 5. 승률 (Hit Ratio)
    hit_ratio = (series > 0).mean()
    
    return {
        'Ann. Return': ann_ret,
        'Ann. Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'MDD': mdd,
        'Hit Ratio': hit_ratio
    }

# ---------------------------------------------------------
# 2. 전체 포트폴리오 성과 분석 실행
# ---------------------------------------------------------
# df_raw가 자네가 제공한 데이터프레임 변수라고 가정
results = {}

for col in df_raw.columns:
    results[col] = calculate_performance(df_raw[col])

# 결과를 DataFrame으로 변환 및 포맷팅
perf_df = pd.DataFrame(results)

# 보기 좋게 포맷팅 (소수점 처리)
# 수익률, 변동성, MDD, 승률은 퍼센트(%)로 표시, 샤프지수는 소수점으로 표시
formatted_perf = perf_df.copy()
formatted_perf.loc[['Ann. Return', 'Ann. Volatility', 'MDD', 'Hit Ratio']] = \
    formatted_perf.loc[['Ann. Return', 'Ann. Volatility', 'MDD', 'Hit Ratio']].applymap(lambda x: f"{x*100:.2f}%")

formatted_perf.loc['Sharpe Ratio'] = formatted_perf.loc['Sharpe Ratio'].apply(lambda x: f"{x:.4f}")

print("=== Portfolio Performance Summary ===")
display(formatted_perf)